# Relative word frequencies

In [ ]:
from tok_preparer.src.settings import tmp_db

In [ ]:
import plotly.offline as pyo
import plotly.graph_objs as go

from itertools import pairwise

import sqlite3

In [ ]:
queries = {'kvinna 1': ['kvinna_1'],
           'kvinna 2': ['kvinna_2'],
           'kvinna 3': ['kvinna_3'],
           'kvinna all': ['kvinna_1', 'kvinna_2', 'kvinna_3']}

In [ ]:
if not tmp_db.exists():
    raise FileNotFoundError(f"Database file not found at {tmp_db}. Please run the data preparation script first.")

In [ ]:
pyo.init_notebook_mode(connected=False)

In [ ]:
def enable_plotly_in_cell():

  import IPython
  from plotly.offline import init_notebook_mode
  display(IPython.core.display.HTML('''<script src="/static/components/requirejs/require.js"></script>'''))
  init_notebook_mode(connected=False)


In [ ]:
from collections import defaultdict


## Looking at incidence of keyword groups

# Relative word frequencies

Since the annual number of utterances shift greatly between 1919 and 1920
-- whereas the number uttered words remains fairly stable -- we need to look
at relative word frequencies instead.


In [ ]:
"""
First, let's recreate the decomposition on utterance level per year
to make sure that the methodology is sound.
"""

with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()
    cur.execute(
    """
    select year,
        count(*) as uc,
        sum(kvinna_1 or kvinna_2 or kvinna_3) as k,
        sum(kvinna_1) as k1,
        sum(kvinna_2) as k2,
        sum(kvinna_3) as k3
    from
        utterance
    group by year
    """)
    results = cur.fetchall()
    x, y_uc, y_k, y_k1, y_k2, y_k3 = zip(*results)
    lines = [go.Scatter(mode='lines', x=x, y=y_uc, name='utterance count'),
             go.Scatter(mode='lines', x=x, y=y_k, name='kvinna'),
             go.Scatter(mode='lines', x=x, y=y_k1, name='kvinna1'),
             go.Scatter(mode='lines', x=x, y=y_k2, name='kvinna2'),
             go.Scatter(mode='lines', x=x, y=y_k3, name='kvinna3'),
            ]
    pyo.iplot({
        'data':lines,
                    'layout':{
                'title':{
                    'text': 'Sanity check: Recreating the "Counting Utterances with different keyword compositions."',
                    'xanchor': 'center',
                    'x':0.5
                }, }})

In [ ]:
# Perfect.

"""
Now we can look at utterance frequencies and relative utterance frequencies.
"""

with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()
    cur.execute(
    """
      select
        year,
        count(*) as uc,
        sum(kvinna_1 or kvinna_2 or kvinna_3) as k,
        cast(sum(kvinna_1 or kvinna_2 or kvinna_3) as real) / count(*) as kr,
        sum(kvinna_1) as k1,
        cast(sum(kvinna_1) as real) / count(*) as k1r,
        sum(kvinna_2) as k2,
        cast(sum(kvinna_2) as real) / count(*) as k2r,
        sum(kvinna_3) as k3,
        cast(sum(kvinna_3) as real) / count(*) as k3r
      from
        utterance
      group by
        year
    """)
    results = cur.fetchall()
    x, y_uc, y_k,y_kr, y_k1, y_k1r, y_k2,y_k2r, y_k3, y_k3r = zip(*results)
    lines = [
            go.Scatter(mode='lines', x=x, y=y_uc, name='utterance count'),
             go.Scatter(mode='lines', x=x, y=y_k, name='kvinna'),
             go.Scatter(mode='lines', x=x, y=y_k1, name='kvinna1'),
             go.Scatter(mode='lines', x=x, y=y_k2, name='kvinna2'),
             go.Scatter(mode='lines', x=x, y=y_k3, name='kvinna3'),
            ]
    pyo.iplot({
        'data':lines,
                              'layout':{
                'title':{
                    'text': 'Counting keywords by category.',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )

    lines = [
             go.Scatter(mode='lines', x=x, y=y_kr, name='kvinna'),
             go.Scatter(mode='lines', x=x, y=y_k1r, name='kvinna1'),
             go.Scatter(mode='lines', x=x, y=y_k2r, name='kvinna2'),
             go.Scatter(mode='lines', x=x, y=y_k3r, name='kvinna3'),
            ]
    pyo.iplot({
        'data':lines,
                              'layout':{
                'title':{
                    'text': 'Relative keyword frequencies by category.',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )

In [ ]:
# Interesting that no big differences show up here.
"Let's add the gender dimension."
with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()
    cur.execute(
    """
      select
        year,
        gender,
        count(*) as uc,
        sum(kvinna_1 or kvinna_2 or kvinna_3) as k,
        cast(sum(kvinna_1 or kvinna_2 or kvinna_3) as real) / count(*) as kr,
        sum(kvinna_1) as k1,
        cast(sum(kvinna_1) as real) / count(*) as k1r,
        sum(kvinna_2) as k2,
        cast(sum(kvinna_2) as real) / count(*) as k2r,
        sum(kvinna_3) as k3,
        cast(sum(kvinna_3) as real) / count(*) as k3r
      from
        utterance
      where gender is not null
      group by
        year, gender
      order by year, gender
    """)
    results = cur.fetchall()

    by_gender = defaultdict(list)
    for row in results:
        year, gender, *values = row
        by_gender[gender].append((year, *values))


    lines = []
    relines = []
    for gender, data in by_gender.items():

        x, y_uc, y_k,y_kr, y_k1, y_k1r, y_k2,y_k2r, y_k3, y_k3r = zip(*data)


        lines += [
              go.Scatter(mode='lines', x=x, y=y_uc, name=f'{gender}: utterance count'),
              go.Scatter(mode='lines', x=x, y=y_k, name=f'{gender}: kvinna'),
              go.Scatter(mode='lines', x=x, y=y_k1, name=f'{gender}: kvinna1'),
              go.Scatter(mode='lines', x=x, y=y_k2, name=f'{gender}: kvinna2'),
              go.Scatter(mode='lines', x=x, y=y_k3, name=f'{gender}: kvinna3'),
              ]

        relines += [
              go.Scatter(mode='lines', x=x, y=y_kr, name=f'{gender}: kvinna'),
              go.Scatter(mode='lines', x=x, y=y_k1r, name=f'{gender}: kvinna1'),
              go.Scatter(mode='lines', x=x, y=y_k2r, name=f'{gender}: kvinna2'),
              go.Scatter(mode='lines', x=x, y=y_k3r, name=f'{gender}: kvinna3'),
              ]

    pyo.iplot({
        'data':lines
    ,
                              'layout':{
                'title':{
                    'text': 'Counting keyword categories by gender.',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )
    pyo.iplot({
        'data':relines,
                              'layout':{
                'title':{
                    'text': 'Relative keyword categories frequencies by gender.',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )

In [ ]:
# Big difference here. Interesting.
# Interesting that no big differences show up here.
"Let's add the party dimension."
with sqlite3.connect(tmp_db) as conn:
    cur = conn.cursor()
    cur.execute(
    """
      select
        year,
        party,
        count(*) as uc,
        sum(kvinna_1 or kvinna_2 or kvinna_3) as k,
        cast(sum(kvinna_1 or kvinna_2 or kvinna_3) as real) / count(*) as kr,
        sum(kvinna_1) as k1,
        cast(sum(kvinna_1) as real) / count(*) as k1r,
        sum(kvinna_2) as k2,
        cast(sum(kvinna_2) as real) / count(*) as k2r,
        sum(kvinna_3) as k3,
        cast(sum(kvinna_3) as real) / count(*) as k3r
      from
        utterance
      where party is not null
      group by
        year, party
      order by year, party
    """)
    results = cur.fetchall()

    by_party = defaultdict(list)
    for row in results:
        year, party, *values = row
        by_party[party].append((year, *values))


    k = []
    k1 = []
    k2 = []
    k3 = []
    for party, data in by_party.items():

        x, y_uc, y_k,y_kr, y_k1, y_k1r, y_k2,y_k2r, y_k3, y_k3r = zip(*data)



        k.append(go.Scatter(mode='lines', x=x, y=y_kr, name=f'{party}'),)
        k1.append(go.Scatter(mode='lines', x=x, y=y_k1r, name=f'{party}'),)
        k2.append(go.Scatter(mode='lines', x=x, y=y_k2r, name=f'{party}'),)
        k3.append(go.Scatter(mode='lines', x=x, y=y_k3r, name=f'{party}'),)


    pyo.iplot({
        'data':k    ,
        'layout':{
                'title':{
                    'text': 'Relative frequencies of "kvinna" by party',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )
    pyo.iplot({
        'data':k1    ,
        'layout':{
                'title':{
                    'text': 'Relative frequencies of "kvinna 1" by party',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )
    pyo.iplot({
        'data':k2    ,
        'layout':{
                'title':{
                    'text': 'Relative frequencies of "kvinna 2" by party',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )
    pyo.iplot({
        'data':k3    ,
        'layout':{
                'title':{
                    'text': 'Relative frequencies of "kvinna 3" by party',
                    'xanchor': 'center',
                    'x':0.5
                }, }}
    )

In [ ]:
def boxes(lower=1900, upper=1940):
    with sqlite3.connect(tmp_db) as conn:

        cur = conn.cursor()

        genders = [ans[0] for ans in cur.execute('select distinct gender from utterance')]

        for gender in genders:
            ark = cur.execute(
                f"""select who, party,
        cast(sum(kvinna_1 or kvinna_2 or kvinna_3) as real) / count(*) as kr,
        cast(sum(kvinna_1) as real) / count(*) as k1r,
        cast(sum(kvinna_2) as real) / count(*) as k2r,
        cast(sum(kvinna_3) as real) / count(*) as k3r
        from utterance
        where gender == "{gender}" and year between {lower} and {upper} group by who
                """).fetchall()
            if ark == []:
                continue

            who, x, yk, yk1, yk2, yk3 = zip(*ark)
            yield go.Bar(x=x, y=yk), go.Bar(x=x, y=yk1), go.Bar(x=x, y=yk2), go.Bar(x=x, y=yk3)

years = list(range(1900, 1941, 10))
full = [(1900, 1940)] + list(pairwise(years))

for start, end in full:
    k = []
    k1 = []
    k2 = []
    k3 = []
    for bk, bk1, bk2, bk3 in boxes(start, end):
        k.append(bk)
        k1.append(bk1)
        k2.append(bk2)
        k3.append(bk3)
    for name, lines in [('kvinna', k), ('kvinna 1', k1), ('kvinna 2', k2), ('kvinna 3', k3)]:
      pyo.iplot({'data' : k,
          'layout':{
              'title':{
                  'text': name + ' ' + str(start) + '-' + str(end),
                  'xanchor': 'center',
                  'x':0.5
              }, }})